# 🌾 KisanAgent — GRPO Training with Unsloth

**Meta-PyTorch OpenEnv Hackathon Finale 2026 — Theme 3: World Modeling**

This notebook trains a Qwen2.5 model using GRPO (Group Relative Policy Optimization) on the KisanAgent RL environment.

## Cell 1 — Install Dependencies

In [ ]:
%%capture
import sys
import os

# Detect Environment
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running in {'Colab' if IN_COLAB else 'Local PC'} environment.")

!pip install unsloth trl gymnasium torch matplotlib seaborn requests pyyaml openenv-core datasets
!pip install --upgrade "accelerate>=0.26.0"

if IN_COLAB:
    REPO_URL = "https://github.com/Gouravbirwaz/meta-hackathon-final-idea"
    REPO_NAME = "meta-hackathon-final-idea"
    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}
    sys.path.insert(0, f"/content/{REPO_NAME}")
else:
    sys.path.insert(0, os.getcwd())

print("Dependencies installed and paths configured.")

## Cell 2 — Initialize Environment

In [ ]:
import torch
from server.app import KisanEnvironment
from env.models import KisanAction

env = KisanEnvironment()
print("KisanEnvironment (OpenEnv) initialized successfully!")

## Cell 3 — Random Agent Baseline

In [ ]:
import random
import numpy as np

DECISIONS = ['irrigate', 'fertilize', 'spray_pesticide', 'sell_now', 'hold_crop', 'apply_scheme', 'take_loan', 'do_nothing']

def run_random_episode(difficulty='medium', seed=42):
    obs = env.reset(difficulty=difficulty, seed=seed)
    for _ in range(90):
        action = KisanAction(farm_decision=random.choice(DECISIONS), reasoning='Random baseline agent')
        obs = env.step(action=action)
        if obs.done:
            meta = obs.metadata or {}
            return meta.get('final_scores', {}).get('composite_score', 0.0)
    return 0.0

print("Calculating Random Baseline (10 episodes)...")
baseline_scores = [run_random_episode('medium', seed=i) for i in range(10)]
RANDOM_BASELINE_SCORE = np.mean(baseline_scores)
print(f"Random Baseline Composite Score: {RANDOM_BASELINE_SCORE:.4f}")

## Cell 4 — Load Model (Unsloth)

In [ ]:
from unsloth import FastLanguageModel
import torch

if IN_COLAB:
    MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
    print("Colab detected: Loading Qwen2.5-7B...")
else:
    MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
    print("Local PC detected: Loading Qwen2.5-1.5B for 4GB VRAM compatibility...")

MAX_SEQ_LENGTH = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print(f"Model {MODEL_NAME} loaded with LoRA adapters.")

## Cell 5 — GRPO Reward Function

In [ ]:
import json as _json

SYSTEM_PROMPT = (
    "You are KisanAgent. Maximize net income across 90 days. "
    "Respond ONLY in JSON: {\"reasoning\": \"...\", \"tool_to_call\": \"tool_name or null\", \"farm_decision\": \"decision or null\"}"
)

def kisan_reward_fn(completions, prompts=None, **kwargs):
    rewards = []
    for completion in completions:
        try:
            text = completion if isinstance(completion, str) else completion[-1].get("content", "")
            text = text.strip()
            if not text.startswith("{"): text = "{" + text
            if not text.endswith("}"): text = text + "}"
            parsed = _json.loads(text)
            action = KisanAction(
                farm_decision=parsed.get("farm_decision") if parsed.get("farm_decision") != "null" else None,
                tool_name=parsed.get("tool_to_call") if parsed.get("tool_to_call") != "null" else None,
                reasoning=parsed.get("reasoning", "")
            )
            env.reset(difficulty='medium', seed=0)
            obs = env.step(action=action)
            reward = float(obs.reward or 0.0)
        except Exception:
            reward = 0.0
        rewards.append(reward)
    return rewards
print("Multi-dimensional Reward Function initialized.")

## Cell 6 — GRPO Training Loop

In [ ]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

CURRICULUM = [(0, 10, 'easy'), (10, 20, 'medium'), (20, 30, 'hard')]
all_prompts = []
for _, _, d in CURRICULUM:
    for ep in range(10):
        obs = env.reset(difficulty=d, seed=ep)
        for day_jump in [0, 25, 50, 75]:
            p = f"Day {obs.day}: Stage {obs.crop_stage}, Balance {obs.bank_balance_inr:.0f}"
            all_prompts.append({'prompt': [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': p},
            ]})

trainer_args = GRPOConfig(
    output_dir="checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=2 if IN_COLAB else 1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_prompt_length=256,
    max_completion_length=128,
    logging_steps=1,
    learning_rate=5e-6,
    save_steps=50,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[kisan_reward_fn],
    args=trainer_args,
    train_dataset=Dataset.from_list(all_prompts),
    processing_class=tokenizer,
)
print(f"Starting GRPO training on {len(all_prompts)} prompts...")
trainer.train()

## Cell 7 — Reward Progress Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import glob

sns.set_theme(style="whitegrid")
checkpoint_dirs = sorted(glob.glob('checkpoints/checkpoint-*'), key=lambda p: int(p.split('-')[-1]))
if checkpoint_dirs:
    with open(os.path.join(checkpoint_dirs[-1], 'trainer_state.json')) as f:
        state = _json.load(f)
    logs = [l for l in state['log_history'] if 'rewards/kisan_reward_fn/mean' in l]
    steps = [l['step'] for l in logs]
    rewards = [l['rewards/kisan_reward_fn/mean'] for l in logs]
    
    plt.figure(figsize=(10, 5))
    plt.title("KisanAgent — GRPO Training Progress (World Modeling)", fontsize=14)
    plt.axhline(y=RANDOM_BASELINE_SCORE, color='red', linestyle='--', label=f'Random Baseline ({RANDOM_BASELINE_SCORE:.2f})')
    plt.scatter(steps, rewards, alpha=0.3, s=10, color='blue', label='Batch Mean Reward')
    if len(rewards) > 5:
        rolling = np.convolve(rewards, np.ones(5)/5, mode='valid')
        plt.plot(steps[len(steps)-len(rolling):], rolling, color='darkblue', linewidth=2, label='Rolling Avg')
    plt.xlabel("Training Steps")
    plt.ylabel("Composite Reward Score")
    plt.legend()
    plt.ylim(0, 1.1)
    plt.show()
else:
    print("No training logs found.")

## Cell 8 — Export to GGUF

In [ ]:
print("Exporting model to GGUF (Q4_K_M) for Agent deployment...")
model.save_pretrained_gguf("kisan_agent_final", tokenizer, quantization_method="q4_k_m")
print("✅ Saved: kisan_agent_final-unsloth.Q4_K_M.gguf")